# Análise Exploratória de Dados (EDA) — Risco de Diabetes

**Projeto Glicuidado — A3 Inteligência Artificial (USJT, 1º Semestre 2026)**

Dataset: **Pima Indians Diabetes Database** (768 registros, 8 features clínicas + alvo binário).

Objetivo desta EDA:
1. Entender a distribuição das variáveis e do alvo.
2. Identificar valores ausentes mascarados como `0` (impossíveis fisiologicamente).
3. Analisar correlações e a relação de cada feature com o diagnóstico.
4. Justificar as decisões de pré-processamento usadas no treino.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

CSV_PATH = os.path.join("..", "data", "diabetes.csv")
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

In [ ]:
# Tipos, contagens e estatísticas descritivas
df.info()
df.describe().T

## 1. Distribuição do alvo

O dataset é **desbalanceado**: há mais casos sem diabetes do que com. Isso justifica
avaliar o modelo com **precision, recall, F1 e ROC-AUC** — e não apenas acurácia.

In [ ]:
print(df["diabetes"].value_counts(normalize=True).round(3))
sns.countplot(data=df, x="diabetes")
plt.title("Distribuição do alvo (0 = sem diabetes, 1 = com diabetes)")
plt.show()

## 2. Valores ausentes mascarados como zero

Em colunas clínicas, o valor `0` é fisiologicamente impossível (ninguém tem glicemia,
pressão arterial ou IMC iguais a zero). Esses zeros são, na verdade, **dados ausentes**.
No pré-processamento, eles são convertidos para `NaN` e imputados pela **mediana**.

In [ ]:
colunas_zero_invalido = ["glicemia", "pressao_arterial", "dobra_cutanea", "insulina", "imc"]
ausentes = {c: int((df[c] == 0).sum()) for c in colunas_zero_invalido}
pd.Series(ausentes).sort_values(ascending=False).plot(kind="bar", color="#E0444B")
plt.title("Quantidade de zeros (valores ausentes) por coluna")
plt.ylabel("Registros com valor 0")
plt.show()
ausentes

## 3. Distribuição das features

In [ ]:
features = ["gestacoes", "glicemia", "pressao_arterial", "dobra_cutanea",
            "insulina", "imc", "hist_familiar", "idade"]
df[features].hist(figsize=(14, 9), bins=25, color="#2F6FED")
plt.suptitle("Histogramas das features")
plt.tight_layout()
plt.show()

## 4. Correlação entre variáveis

A **glicemia** costuma ser a variável mais correlacionada com o diagnóstico, seguida
por IMC e idade — coerente com a literatura médica sobre diabetes tipo 2.

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="Blues")
plt.title("Matriz de correlação")
plt.show()

df.corr(numeric_only=True)["diabetes"].drop("diabetes").sort_values(ascending=False)

## 5. Relação das features com o diagnóstico

Boxplots comparando cada feature entre quem tem e não tem diabetes.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.flat, features):
    sns.boxplot(data=df, x="diabetes", y=col, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## Conclusões da EDA

- **Alvo desbalanceado** (~65% sem diabetes / ~35% com) → usar métricas além da acurácia.
- **Zeros inválidos** em glicemia, pressão, dobra cutânea, insulina e IMC → tratados como
  ausentes e imputados pela mediana (robusta a outliers).
- **Glicemia, IMC e idade** são as variáveis mais associadas ao diagnóstico.
- Features em **escalas muito diferentes** (ex.: insulina vs. função pedigree) →
  justifica a **padronização (StandardScaler)** para modelos sensíveis à escala.

Essas decisões estão implementadas em `utils/preprocessamento.py` e aplicadas no treino
(`models/treino_modelo.py`).